In [9]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_csv('health_data/health_application.csv')
df_id_authors = df[['ID', 'Authors']].copy()

df_id_authors

,ID,Authors
0,119,"Bui, Nam and Pham, Nhat and Barnitz, Jessica J..."
1,120,"R\""{o}ddiger, Tobias and Beigl, Michael and He..."
2,121,E. Nemati; S. Zhang; T. Ahmed; M. M. Rahman; J...
3,122,S. Zhang; E. Nemati; T. Ahmed; M. M. Rahman; J...
4,123,N. Nguyen; A. Chakma; N. Roy
5,124,A. Boukhayma; A. Barison; S. Haddad; A. Caizzone
6,125,S. -j. Jung; J. Ryu; W. Kim; S. Lee; J. Kim; H...
7,126,J. Meneses; O. Miranda; I. Sanchez; C. Álvarez...
8,127,J. Juez; D. Henao; F. Segura; R. Gómez; M. Le ...
9,128,K. -J. Kim; K. -T. Lim; J. w. Baek; M. Shin


In [ ]:
# Build exact co-author matrix from comma-separated author strings
ids = df_id_authors['ID'].to_numpy(dtype=int)
ids = np.unique(ids)
ids = np.sort(ids)

coauthor_matrix = pd.DataFrame(0, index=ids, columns=ids)

def normalize_name(name: str) -> str:
    return ' '.join(name.strip().lower().split())

def to_author_set(value) -> set:
    if isinstance(value, list):
        names = value
    elif isinstance(value, str):
        names = [n for n in (x.strip() for x in value.split(',')) if n]
    else:
        names = []
    return {normalize_name(n) for n in names}

id_to_authors = {int(row['ID']): to_author_set(row['Authors']) for _, row in df_id_authors.iterrows()}

for a in range(len(ids)):
    id_i = ids[a]
    authors_i = id_to_authors.get(id_i, set())
    if not authors_i:
        continue
    for b in range(a + 1, len(ids)):
        id_j = ids[b]
        authors_j = id_to_authors.get(id_j, set())
        if authors_j and authors_i.intersection(authors_j):
            coauthor_matrix.loc[id_i, id_j] = 1
            coauthor_matrix.loc[id_j, id_i] = 1

coauthor_matrix.to_csv('interconnections_datasets/coauthor_matrix_health_test.csv')
coauthor_matrix.head()

,119,120,121,122,123,124,125,126,127,128,...,159,160,161,162,163,164,165,166,167,168
119,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
120,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
121,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
122,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
123,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
